In [ ]:
!pip install tensorflow-datasets -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras.datasets import mnist

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

print("TensorFlow Version:", tf.__version__)







# LOAD MNIST DATASET



print("\nLOADING MNIST DATASET...")

(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = mnist.load_data()

print("MNIST Train Shape:", X_train_mnist.shape)

print("MNIST Test Shape :", X_test_mnist.shape)







# LOAD EMNIST LETTERS DATASET



# REMOVE CORRUPTED CACHE FILES


print("\nLOADING EMNIST LETTERS DATASET...")

train_ds = tfds.load(
    'emnist/letters',
    split='train',
    as_supervised=True
)

test_ds = tfds.load(
    'emnist/letters',
    split='test',
    as_supervised=True
)

# TAKE SMALLER DATA FOR BEGINNERS

X_train_emnist = []
y_train_emnist = []

for image, label in tfds.as_numpy(train_ds.take(20000)):

    X_train_emnist.append(image)

    y_train_emnist.append(label)

X_test_emnist = []
y_test_emnist = []

for image, label in tfds.as_numpy(test_ds.take(4000)):

    X_test_emnist.append(image)

    y_test_emnist.append(label)

X_train_emnist = np.array(X_train_emnist)

y_train_emnist = np.array(y_train_emnist)

X_test_emnist = np.array(X_test_emnist)

y_test_emnist = np.array(y_test_emnist)

print("EMNIST Train Shape:", X_train_emnist.shape)

print("EMNIST Test Shape :", X_test_emnist.shape)







X_train_mnist = X_train_mnist.reshape(-1,28,28,1)

X_test_mnist = X_test_mnist.reshape(-1,28,28,1)

X_train_mnist = X_train_mnist / 255.0

X_test_mnist = X_test_mnist / 255.0

y_train_mnist = to_categorical(y_train_mnist, 10)

y_test_mnist = to_categorical(y_test_mnist, 10)

print("MNIST PREPROCESSING COMPLETED")



X_train_emnist = X_train_emnist / 255.0

X_test_emnist = X_test_emnist / 255.0

# LABELS START FROM 1

y_train_emnist = y_train_emnist - 1

y_test_emnist = y_test_emnist - 1

y_train_emnist = to_categorical(y_train_emnist, 26)

y_test_emnist = to_categorical(y_test_emnist, 26)

print("EMNIST PREPROCESSING COMPLETED")


mnist_classes = np.argmax(y_train_mnist, axis=1)

plt.figure(figsize=(10,5))

sns.countplot(x=mnist_classes)

plt.title("MNIST Class Distribution")

plt.show()


mnist_model = Sequential()

mnist_model.add(
    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(28,28,1)
    )
)

mnist_model.add(BatchNormalization())

mnist_model.add(MaxPooling2D((2,2)))

mnist_model.add(Dropout(0.25))

mnist_model.add(
    Conv2D(
        64,
        (3,3),
        activation='relu'
    )
)

mnist_model.add(BatchNormalization())

mnist_model.add(MaxPooling2D((2,2)))

mnist_model.add(Dropout(0.25))

mnist_model.add(Flatten())

mnist_model.add(Dense(128, activation='relu'))

mnist_model.add(Dropout(0.5))

mnist_model.add(Dense(10, activation='softmax'))

mnist_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mnist_model.summary()


early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


history = mnist_model.fit(
    X_train_mnist,
    y_train_mnist,
    validation_split=0.2,
    epochs=10,
    batch_size=128,
    callbacks=[early_stop]
)


train_loss, train_acc = mnist_model.evaluate(
    X_train_mnist,
    y_train_mnist,
    verbose=0
)

test_loss, test_acc = mnist_model.evaluate(
    X_test_mnist,
    y_test_mnist,
    verbose=0
)

print("\nTRAINING ACCURACY :", round(train_acc*100,2), "%")

print("TESTING ACCURACY  :", round(test_acc*100,2), "%")

gap = train_acc - test_acc

print("OVERFITTING GAP   :", round(gap*100,2), "%")

if gap < 0.05:

    print("Generalization: Excellent ✅")

else:

    print("Overfitting Detected ⚠️")


y_pred = mnist_model.predict(X_test_mnist)

y_pred_classes = np.argmax(y_pred, axis=1)

y_true = np.argmax(y_test_mnist, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()


print(
    classification_report(
        y_true,
        y_pred_classes
    )
)


plt.figure(figsize=(10,5))

plt.plot(history.history['accuracy'])

plt.plot(history.history['val_accuracy'])

plt.title("Training vs Validation Accuracy")

plt.xlabel("Epochs")

plt.ylabel("Accuracy")

plt.legend(['Train','Validation'])

plt.show()


plt.figure(figsize=(10,5))

plt.plot(history.history['loss'])

plt.plot(history.history['val_loss'])

plt.title("Training vs Validation Loss")

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.legend(['Train','Validation'])

plt.show()


mnist_model.save("handwritten_character_model.keras")

print("MODEL SAVED SUCCESSFULLY")





















In [ ]:
#EMNIST DATA SET
#CELL 1 — INSTALL LIBRARIES
!pip install tensorflow tensorflow-datasets -q
#CELL 2 — IMPORT LIBRARIES
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

print("TensorFlow Version:", tf.__version__)
#CELL 3 — LOAD EMNIST LETTERS DATASET
train_ds = tfds.load(
    'emnist/letters',
    split='train',
    as_supervised=True
)

test_ds = tfds.load(
    'emnist/letters',
    split='test',
    as_supervised=True
)

X_train = []
y_train = []

for image, label in tfds.as_numpy(train_ds.take(50000)):

    X_train.append(image)

    y_train.append(label)

X_test = []
y_test = []

for image, label in tfds.as_numpy(test_ds.take(10000)):

    X_test.append(image)

    y_test.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = np.array(X_test)
y_test = np.array(y_test)

print("Training Shape:", X_train.shape)

print("Testing Shape :", X_test.shape)
#CELL 4 — DISPLAY SAMPLE LETTERS
plt.figure(figsize=(10,5))

for i in range(10):

    plt.subplot(2,5,i+1)

    plt.imshow(X_train[i].squeeze(), cmap='gray')

    plt.title(chr(y_train[i] + 96))

    plt.axis('off')

plt.show()
#CELL 5 — PREPROCESSING
X_train = X_train / 255.0
X_test = X_test / 255.0

y_train = y_train - 1
y_test = y_test - 1

y_train = to_categorical(y_train,26)
y_test = to_categorical(y_test,26)

print("Preprocessing Completed")
#CELL 6 — CLASS DISTRIBUTION
classes = np.argmax(y_train, axis=1)

plt.figure(figsize=(12,5))

sns.countplot(x=classes)

plt.title("EMNIST Letter Distribution")

plt.show()
#CELL 7 — BUILD IMPROVED CNN MODEL
model = Sequential()

model.add(
    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(28,28,1)
    )
)

model.add(BatchNormalization())

model.add(
    Conv2D(
        32,
        (3,3),
        activation='relu'
    )
)

model.add(MaxPooling2D((2,2)))

model.add(Dropout(0.25))

model.add(
    Conv2D(
        64,
        (3,3),
        activation='relu'
    )
)

model.add(BatchNormalization())

model.add(
    Conv2D(
        64,
        (3,3),
        activation='relu'
    )
)

model.add(MaxPooling2D((2,2)))

model.add(Dropout(0.25))

model.add(Flatten())

model.add(Dense(256, activation='relu'))

model.add(BatchNormalization())

model.add(Dropout(0.5))

model.add(Dense(26, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
#CELL 8 — CALLBACKS
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2
)
#CELL 9 — TRAIN MODEL
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=128,
    callbacks=[early_stop, reduce_lr]
)
#CELL 10 — EVALUATION
train_loss, train_acc = model.evaluate(
    X_train,
    y_train,
    verbose=0
)

test_loss, test_acc = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTRAINING ACCURACY :", round(train_acc*100,2), "%")

print("TESTING ACCURACY  :", round(test_acc*100,2), "%")

gap = train_acc - test_acc

print("OVERFITTING GAP   :", round(gap*100,2), "%")

if gap < 0.05:

    print("Generalization: Excellent ✅")

else:

    print("Overfitting Detected ⚠️")
#CELL 11 — CONFUSION MATRIX
y_pred = model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)

y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(14,10))

sns.heatmap(
    cm,
    cmap='Blues'
)

plt.title("EMNIST Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()
#CELL 12 — CLASSIFICATION REPORT
print(
    classification_report(
        y_true,
        y_pred_classes
    )
)
#CELL 13 — ACCURACY GRAPH
plt.figure(figsize=(10,5))

plt.plot(history.history['accuracy'])

plt.plot(history.history['val_accuracy'])

plt.title("Training vs Validation Accuracy")

plt.xlabel("Epochs")

plt.ylabel("Accuracy")

plt.legend(['Train','Validation'])

plt.show()
#CELL 14 — LOSS GRAPH
plt.figure(figsize=(10,5))

plt.plot(history.history['loss'])

plt.plot(history.history['val_loss'])

plt.title("Training vs Validation Loss")

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.legend(['Train','Validation'])

plt.show()
#CELL 15 — SAVE MODEL
model.save("emnist_letters_model.keras")

print("EMNIST MODEL SAVED SUCCESSFULLY")

In [ ]:
# ============================================================
# PREDICTIONS
# ============================================================

y_pred = model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)

y_true = np.argmax(y_test, axis=1)

print("Prediction Shape :", y_pred_classes.shape)

print("True Label Shape :", y_true.shape)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred_classes
)

plt.figure(figsize=(14,12))

sns.heatmap(
    cm,
    cmap='Blues'
)

plt.title("EMNIST Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        y_true,
        y_pred_classes
    )
)


# ============================================================
# ACCURACY GRAPH
# ============================================================

plt.figure(figsize=(10,5))

plt.plot(history.history['accuracy'])

plt.plot(history.history['val_accuracy'])

plt.title("Training vs Validation Accuracy")

plt.xlabel("Epochs")

plt.ylabel("Accuracy")

plt.legend(['Train','Validation'])

plt.show()


# ============================================================
# LOSS GRAPH
# ============================================================

plt.figure(figsize=(10,5))

plt.plot(history.history['loss'])

plt.plot(history.history['val_loss'])

plt.title("Training vs Validation Loss")

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.legend(['Train','Validation'])

plt.show()


# ============================================================
# SAVE MODEL
# ============================================================

model.save("emnist_letters_model.keras")

print("MODEL SAVED SUCCESSFULLY")

In [ ]:
from google.colab import files
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# LOAD MODEL (FIXED)
emnist_model = tf.keras.models.load_model("emnist_letters_model.keras")

# UPLOAD FILE
uploaded = files.upload()

for img_path in uploaded.keys():

    # READ IMAGE
    img = cv2.imread(img_path)

    # GRAYSCALE
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # INVERT (important for EMNIST style)
    gray = 255 - gray

    # RESIZE
    gray = cv2.resize(gray, (28, 28))

    # NORMALIZE
    gray = gray / 255.0

    # SHOW IMAGE
    plt.imshow(gray, cmap='gray')
    plt.title("Processed Image")
    plt.axis('off')
    plt.show()

    # RESHAPE
    gray = gray.reshape(1, 28, 28, 1)

    # PREDICT
    prediction = emnist_model.predict(gray)

    predicted_class = np.argmax(prediction)

    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

    print("Predicted Letter:", letters[predicted_class])

In [ ]:
from google.colab import files
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# LOAD MNIST MODEL (.keras file required)
mnist_model = tf.keras.models.load_model("handwritten_character_model.keras")

uploaded = files.upload()

for img_path in uploaded.keys():

    img = cv2.imread(img_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    gray = 255 - gray   # invert (important for handwritten digits)

    gray = cv2.resize(gray, (28, 28))

    gray = gray / 255.0

    plt.imshow(gray, cmap='gray')
    plt.title("MNIST Processed Image")
    plt.axis('off')
    plt.show()

    gray = gray.reshape(1, 28, 28, 1)

    prediction = mnist_model.predict(gray)

    predicted_class = np.argmax(prediction)

    print("Predicted Digit:", predicted_class)